### Fetch Transcript

In [1]:
from youtube_transcript_api import YouTubeTranscriptApi
import re
from sentence_transformers import SentenceTransformer
import spacy
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
import numpy as np
from typing import List, Any
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
load_dotenv()

c:\Users\apaks\projects\YT-RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [46]:
# fetch video id from url
def fetch_video_id(url:str)-> str:
    try:
        pattern = re.compile(
            r"(?:youtube\.com\/watch\?v=)([a-zA-Z0-9_-]{11})"
        )
        match = re.search(pattern, url)
        return match.group(1) if match else None
    except Exception as e:
        print(f"[INFO] No transcript found. Error: {e}")
        raise

In [74]:
url = "https://www.youtube.com/watch?v=n_3XDVOVraI&t=1341s"
video_id = fetch_video_id(url)
print(video_id)

n_3XDVOVraI


In [75]:
ytt_api = YouTubeTranscriptApi()
fetched_transcript = ytt_api.fetch(video_id = video_id)


In [76]:
fetched_transcript

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='alright so welcome back to another', start=0.0, duration=4.86), FetchedTranscriptSnippet(text='podcast episode on the data Professor', start=1.979, duration=5.161), FetchedTranscriptSnippet(text='YouTube channel uh today we have a', start=4.86, duration=4.02), FetchedTranscriptSnippet(text='special guest on the channel we have', start=7.14, duration=5.34), FetchedTranscriptSnippet(text="Jess Haberman and I think it's best if", start=8.88, duration=5.7), FetchedTranscriptSnippet(text='Jess could you introduce yourself please', start=12.48, duration=4.2), FetchedTranscriptSnippet(text='sure be happy to', start=14.58, duration=4.02), FetchedTranscriptSnippet(text="um I'm a director of learning Solutions", start=16.68, duration=3.96), FetchedTranscriptSnippet(text='at Anaconda', start=18.6, duration=6.36), FetchedTranscriptSnippet(text='um and I actually previously was senior', start=20.64, duration=7.68), FetchedTranscriptSnippet(

In [77]:
transcript:list = []
for snipet in fetched_transcript:
    transcript.append(snipet.text)

In [78]:
text = " ".join(transcript)

In [79]:
text

"alright so welcome back to another podcast episode on the data Professor YouTube channel uh today we have a special guest on the channel we have Jess Haberman and I think it's best if Jess could you introduce yourself please sure be happy to um I'm a director of learning Solutions at Anaconda um and I actually previously was senior Acquisitions editor at O'Reilly media um overseeing sort of the data science and data engineering space um and then Anaconda um we're developing a learning platform to help folks up skill and reskill in data science data engineering machine learning all that great stuff learning python um so yeah I'm really happy to be on the show today yeah awesome yeah and a pleasure to have you yeah so before we get started with other questions I guess it would be awesome to start with the like a short introductory story about your career journey into the data doing mean yeah um so I have a pretty non-technical background I've been in book publishing my entire career uh 

### Chunking

In [80]:
# linguistic preprocessings to get proper sentences

nlp = spacy.load("en_core_web_sm")
doc = nlp(text)
doc

alright so welcome back to another podcast episode on the data Professor YouTube channel uh today we have a special guest on the channel we have Jess Haberman and I think it's best if Jess could you introduce yourself please sure be happy to um I'm a director of learning Solutions at Anaconda um and I actually previously was senior Acquisitions editor at O'Reilly media um overseeing sort of the data science and data engineering space um and then Anaconda um we're developing a learning platform to help folks up skill and reskill in data science data engineering machine learning all that great stuff learning python um so yeah I'm really happy to be on the show today yeah awesome yeah and a pleasure to have you yeah so before we get started with other questions I guess it would be awesome to start with the like a short introductory story about your career journey into the data doing mean yeah um so I have a pretty non-technical background I've been in book publishing my entire career uh s

In [81]:
sentences = [sent.text for sent in doc.sents]

In [82]:
text_splitter = RecursiveCharacterTextSplitter(separators = ["\n\n", "\n", ". ", "? ", ", ", "! ", " ", ""], chunk_size = 700, chunk_overlap = 100, length_function = len)
text_splitter.split_text("".join(sentences))

["alright so welcome back to another podcast episode on the data Professor YouTube channel uh today we have a special guest on the channel we have Jess Habermanand I think it's best if Jess could you introduce yourself please sure be happy to um I'm a director of learning Solutions at Anaconda umand I actually previously was senior Acquisitions editor at O'Reilly media um overseeing sort of the data science and data engineering space um and then Anaconda um we're developing a learning platform to help folks up skill and reskill in data science data engineering machine learning all that great stuff learning pythonum soyeah I'm really happy to be on the show todayyeahawesomeyeah and a pleasure",
 "learning pythonum soyeah I'm really happy to be on the show todayyeahawesomeyeah and a pleasure to have youyeahso before we get started with other questions I guess it would be awesome to start with the like a short introductory story about your career journey into the data doingmeanyeah umso I

#### Semantic chunking

In [83]:
len(sentences)

772

In [84]:
def semantic_chunking(sentences:List[str], similarity_threshold:float = 0.20, max_tokens:int = 500, overlap:int = 1):
    embedder = OpenAIEmbeddings(model = "text-embedding-3-large")
    embeddings = embedder.embed_documents(sentences)
    chunks = []
    current_chunk = []
    current_tokens = 0

    for i, sentence in enumerate(sentences):
        sentence_tokens = len(sentence.split())

        if not current_chunk:
            current_chunk.append(sentence)
            current_tokens += sentence_tokens 
            continue

        # calculate similarity score between sentence and previous sentence
        sim = np.dot(embeddings[i], embeddings[i-1]) / (np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i-1]))

        if sim < similarity_threshold and current_tokens > max_tokens:
            chunks.append(" ".join(current_chunk))

            current_chunk = current_chunk[-overlap:] if overlap > 0 else []
            current_tokens = sum(len(s.split()) for s in current_chunk)

        current_chunk.append(sentence)
        current_tokens += sentence_tokens

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

In [85]:
chunks = semantic_chunking(sentences=sentences, similarity_threshold=0.15, max_tokens=500)

In [86]:
# savings chunks for experimentation
import pickle
with open('test-chunks.pkl', 'wb') as f:
    pickle.dump(chunks, f)

In [87]:
len(chunks)

22

In [88]:
chunks

["alright so welcome back to another podcast episode on the data Professor YouTube channel uh today we have a special guest on the channel we have Jess Haberman and I think it's best if Jess could you introduce yourself please sure be happy to um I'm a director of learning Solutions at Anaconda um and I actually previously was senior Acquisitions editor at O'Reilly media um overseeing sort of the data science and data engineering space um and then Anaconda um we're developing a learning platform to help folks up skill and reskill in data science data engineering machine learning all that great stuff learning python um so yeah I'm really happy to be on the show today yeah awesome yeah and a pleasure to have you yeah so before we get started with other questions I guess it would be awesome to start with the like a short introductory story about your career journey into the data doing mean yeah um so I have a pretty non-technical background I've been in book publishing my entire career uh

In [89]:
# generate embeddings of chunks

embedder = OpenAIEmbeddings(model = "text-embedding-3-large")
embeddings = embedder.embed_documents(chunks)

In [90]:
embeddings = np.array(embeddings)

### Vectorstore

In [91]:
embeddings.shape

(22, 3072)

In [92]:
# create FAISS index

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)

In [93]:
# add embeddings
index.add(embeddings)

In [94]:
# save the index 
faiss.write_index(index, "faiss.index")


In [95]:
# load faiss index
loaded_index = faiss.read_index("faiss.index")

In [96]:
# testing new query
query = "What are ethical limits of tools like chatgpt?"
query_embedding = embedder.embed_documents([query])

query_embedding = np.array(query_embedding)

In [97]:
query_embedding.shape

(1, 3072)

In [98]:
D, I = loaded_index.search(query_embedding, k= 5)

In [99]:
I

array([[6, 5, 4, 7, 8]])

In [100]:
chunks[I[0][0]]

"and I remember there's a quote from Santiago also a data um influencer on Twitter uh he mentioned about like a person not using AI will be replaced with a person using AI tools and yeah as you have mentioned that clearly uh describes like the situation and I've also like found like some guidelines from elsevier publisher um they mentioned that they allow the user chat to Unity but then I saw that more or less for language help right like kind of like in the context of like grammarly or but not to do you know like the actual uh literature writing and yeah that should be done you know like by a human yeah I and I agree with that stance I think it's like let's let's you know let's let people again like use Google use the spell check use grammarly use like these llm tools um but you know check your sources and you know and do the due diligence you would do when you found something on Wikipedia or whatever you know it's like you still need to do that that checking yeah and you know like uh